# Gemma 4 E2B-IT Bio-Safety SAE Training — Colab T4

> **Changelog — 2026-04-26 (v2):** Dataset upgraded: `cais/wmdp-bio-forget-corpus` (24,453 hazard-adjacent biology papers, gated access granted 2026-04-26) replaces the 22-prompt local eval-set fallback. MAX_STEPS increased to 5000. RUN_TAG → `gemma4-e2b-bio-sae-v2`. Balanced corpus: 5,000 hazard + 5,000 benign. This run is expected to sustain L_contrastive (corpus-size bottleneck was the root cause of collapse in v1).
>
> **Changelog — 2026-04-26 (v1):** Replaced broken notebook with version tested end-to-end on Colab T4 (Tesla T4, 15.6 GB VRAM, CUDA 13.0). Key fixes: `pick_layer()` with 5-path fallback + slow-path `named_modules()` scan for Gemma 4 multimodal architecture; `TopKSAE` uses `nn.Linear` instead of `nn.Parameter` for encoder/decoder; `normalize_decoder()` + `project_grad()` enforce unit-sphere decoder constraint each step; training prompts wrapped in Gemma chat template via `tokenizer.apply_chat_template`; mean contrastive loss (cosine sim of mean hazard vs mean benign feature profiles) replaces unstable NT-Xent; `MAX_STEPS=2000`, `BATCH_SIZE=4`. Trained checkpoint uploaded to `Solshine/gemma4-e2b-bio-sae-v1` on HuggingFace.

This notebook implements domain-specific fine-tuning for a TopK Sparse Autoencoder (SAE) on the Gemma 4 E2B-IT model. The goal is to enhance the SAE's representation of biosecurity concepts, specifically distinguishing between benign biological content and hazardous biological knowledge.

**Specifications:**
- **Model:** Gemma 4 E2B-IT (frozen, NF4 4-bit quantization).
- **SAE:** TopK (d_model=1536, d_sae=6144, k=32).
- **Loss:** L = L_recon + λ_sparsity * L_sparsity + λ_contrastive * L_contrastive.
- **Contrastive Objective:** Discriminates between hazard-adjacent and benign biological activations.

**Dataset options (cell [5/7]):**

| Option | Dataset | Tier coverage | Access |
|--------|---------|--------------|--------|
| `"wmdp"` (default) | `cais/wmdp-corpora` (benign) + `cais/wmdp-bio-forget-corpus` (hazard, gated) | 5K hazard + 5K benign | Gated (access required for hazard) |
| `"gated"` | `SolshineCode/biorefusalaudit-gated` — BioRefusalAudit curated eval set | hazard + dual-use + benign (75→10K) | HF gated |
| `"local"` | Local `data/eval_set_public/eval_set_public_v1.jsonl` | all tiers (75 prompts) | Always available |

**Setup:**
1. Change runtime to **T4 GPU**.
2. Add `HF_TOKEN` (write scope) and `WANDB_API_KEY` to Colab 🔑 Secrets.
   - `HF_TOKEN` is required only for the `"gated"` dataset option and checkpoint saves to HF; not needed for the default `"wmdp"` path.
3. Toggle notebook access for both secrets.

In [5]:
# [1/7] GPU + Environment Check
import torch
import subprocess

r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout[:1000] if r.returncode == 0 else "NO GPU found.")

assert torch.cuda.is_available(), "CUDA not available — set Runtime to T4 GPU"
print("\nTorch version:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0))
print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

Sun Apr 26 05:45:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             31W /   70W |     179MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# [2/7] Install Dependencies + Auth
# Install transformers from source: Gemma 4 ('gemma4' model type) requires the latest
# main branch. PyPI releases lag and raise KeyError: 'gemma4' on AutoConfig.
# Fix applied 2026-04-25 after KeyError: 'gemma4' on PyPI transformers install.
!pip install -q git+https://github.com/huggingface/transformers.git accelerate bitsandbytes datasets huggingface_hub wandb sentencepiece

import os, json, time
from google.colab import userdata
from huggingface_hub import login, HfApi
import wandb

for key in ("HF_TOKEN", "GITHUB_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key}: loaded")
    except:
        print(f"{key}: not set")

if os.environ.get("HF_TOKEN"): login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
if os.environ.get("WANDB_API_KEY"): wandb.login(key=os.environ["WANDB_API_KEY"])

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
HF_TOKEN: loaded
GITHUB_TOKEN: loaded
WANDB_API_KEY: loaded


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: caleb-deleeuw to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
# [2b/7] Repo Clone (optional — WMDP/HF-only path works without it)
#
# Priority:
#   1. GITHUB_TOKEN set → authenticated clone (works for private repo)
#   2. No token, HF_TOKEN set → unauthenticated clone attempt; if it fails,
#      continue in HF-only mode (WMDP data from HF, checkpoints to HF)
#   3. Neither token → same unauthenticated attempt with same fallback
#
# REPO_AVAILABLE is set False when clone fails; cell [5/7] skips the
# "local" fallback and forces the WMDP or gated HF path instead.

import os
import subprocess
from pathlib import Path

_REPO   = "SolshineCode/Deleeuw-AI-x-Bio-hackathon"
_BRANCH = "main"
_WORK   = Path("/content/Deleeuw-AI-x-Bio-hackathon")

_token = os.environ.get("GITHUB_TOKEN", "")
_clone_url = (
    f"https://{_token}@github.com/{_REPO}.git" if _token
    else f"https://github.com/{_REPO}.git"
)

REPO_AVAILABLE = False

if _WORK.exists():
    r = subprocess.run(
        ["git", "-C", str(_WORK), "pull", "--ff-only", "origin", _BRANCH],
        capture_output=True, text=True,
    )
    REPO_AVAILABLE = True  # dir exists — usable even if pull had nothing to do
    print("Repo present" + ("; pulled latest." if r.returncode == 0 else "; pull skipped (already up to date or detached HEAD)."))
else:
    r = subprocess.run(
        ["git", "clone", "--branch", _BRANCH, _clone_url, str(_WORK)],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        REPO_AVAILABLE = True
        print(f"Cloned {_REPO} @ {_BRANCH}.")
    else:
        hf_token = os.environ.get("HF_TOKEN", "")
        if hf_token:
            print(f"Clone failed (rc={r.returncode}) — HF_TOKEN is set, continuing in HF-only mode.")
            print("Dataset: WMDP from HuggingFace.  Checkpoints: uploaded to your HF account.")
        else:
            print(f"Clone failed (rc={r.returncode}) and no HF_TOKEN set.")
            print("Add GITHUB_TOKEN or HF_TOKEN to Colab Secrets (🔑) and re-run.")
        print(f"git stderr: {r.stderr.strip()[:300]}")

if REPO_AVAILABLE:
    %cd /content/Deleeuw-AI-x-Bio-hackathon
    print("Working directory: /content/Deleeuw-AI-x-Bio-hackathon")
else:
    print("Working directory: /content  (HF-only mode)")

Cloned SolshineCode/Deleeuw-AI-x-Bio-hackathon @ main.
/content/Deleeuw-AI-x-Bio-hackathon
Working directory: /content/Deleeuw-AI-x-Bio-hackathon


In [8]:
# [3/7] Load Frozen Gemma 4 E2B-IT
import torch.nn as nn
from transformers import AutoTokenizer, BitsAndBytesConfig

HF_MODEL = "google/gemma-4-E2B-it"
LAYER = 17
D_MODEL = 1536

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading {HF_MODEL}...")
# Gemma 4 is a multimodal model; try CausalLM first, fall back to ImageTextToText
try:
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL, quantization_config=bnb_config,
        device_map={"":0}, low_cpu_mem_usage=True
    )
except Exception:
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(
        HF_MODEL, quantization_config=bnb_config,
        device_map={"":0}, low_cpu_mem_usage=True
    )
model.eval()
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
tokenizer.pad_token = tokenizer.eos_token

class ActivationHook:
    def __init__(self): self.last = None
    def __call__(self, module, inputs, outputs):
        # Overwrite (not append) to avoid VRAM accumulation during generation
        self.last = outputs[0].detach() if isinstance(outputs, tuple) else outputs.detach()

def pick_layer(m, layer_idx):
    """Find transformer block list across Gemma 2/3/4 text-only & multimodal layouts.

    Tries common attribute paths first, then falls back to a named_modules() scan
    so a transformers-version refactor (e.g. Gemma 4's model.language_model.layers
    layout) doesn't break activation hooking.
    """
    # Fast path: known layouts in order of likelihood
    for path in (
        "model.language_model",   # Gemma 4 ForConditionalGeneration (current)
        "language_model.model",   # Gemma 3 ForConditionalGeneration (older)
        "language_model",         # some multimodal variants
        "model",                  # text-only ForCausalLM
        "transformer",            # GPT-style fallback
    ):
        obj = m
        try:
            for attr in path.split("."):
                obj = getattr(obj, attr)
        except AttributeError:
            continue
        if hasattr(obj, "layers"):
            return obj.layers[layer_idx]
    # Slow path: scan all submodules for a ModuleList named "layers"
    for name, mod in m.named_modules():
        if name.endswith(".layers") and hasattr(mod, "__getitem__"):
            try:
                return mod[layer_idx]
            except (IndexError, TypeError):
                continue
    raise AttributeError(
        f"Could not locate transformer layer list in {type(m).__name__}; "
        f"top-level children: {[n for n, _ in m.named_children()]}"
    )

hook = ActivationHook()
handle = pick_layer(model, LAYER).register_forward_hook(hook)
print(f"Attached hook at layer {LAYER}")


Loading google/gemma-4-E2B-it...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Attached hook at layer 17


In [9]:
# [4/7] TopK Sparse Autoencoder (SAE) Definition
import torch
import torch.nn as nn
import torch.nn.functional as F

D_MODEL = 1536   # Gemma 4 E2B hidden size (layer 17)
D_SAE   = 6144   # expansion factor 4×
K       = 32     # TopK sparsity

class TopKSAE(nn.Module):
    """TopK Sparse Autoencoder with decoder norm-constraint and pre-activation tracking."""

    def __init__(self, d_model: int = D_MODEL, d_sae: int = D_SAE, k: int = K):
        super().__init__()
        self.k = k
        # Encoder: d_model → d_sae
        self.W_enc = nn.Linear(d_model, d_sae, bias=True)
        # Decoder: d_sae → d_model  (unit-norm columns enforced in normalize_decoder)
        self.W_dec = nn.Linear(d_sae, d_model, bias=True)
        # Initialise decoder columns to unit norm
        with torch.no_grad():
            self.W_dec.weight.data = F.normalize(self.W_dec.weight.data, dim=0)

    def forward(self, x):
        """
        Args:
            x: (batch, d_model) or (seq, d_model) float tensor on CUDA
        Returns:
            x_hat: reconstruction, same shape as x
            z:     post-topk sparse activations  (..., d_sae)
            pre_relu: pre-activation values for sparsity loss  (..., d_sae)
        """
        pre_relu = self.W_enc(x)          # (..., d_sae)
        # TopK: zero out all but the top-k activations per token
        topk_vals, topk_idx = torch.topk(pre_relu, self.k, dim=-1)
        z = torch.zeros_like(pre_relu)
        z.scatter_(-1, topk_idx, F.relu(topk_vals))
        x_hat = self.W_dec(z)             # (..., d_model)
        return x_hat, z, pre_relu

    @torch.no_grad()
    def normalize_decoder(self):
        """Project decoder columns back onto the unit sphere after each gradient step."""
        self.W_dec.weight.data = F.normalize(self.W_dec.weight.data, dim=0)

    @torch.no_grad()
    def project_grad(self):
        """Remove the component of the decoder gradient parallel to each column
        (keeps the optimiser from immediately re-scaling columns away from norm=1)."""
        if self.W_dec.weight.grad is not None:
            w = self.W_dec.weight.data          # (d_model, d_sae)
            g = self.W_dec.weight.grad           # (d_model, d_sae)
            # proj of g onto w (column-wise)
            proj = (g * w).sum(0, keepdim=True) * w
            self.W_dec.weight.grad -= proj

# Instantiate SAE on CUDA
sae = TopKSAE(D_MODEL, D_SAE, K).cuda().float()
print(f"SAE: d_model={D_MODEL}, d_sae={D_SAE}, k={K}")
print(f"Parameters: {sum(p.numel() for p in sae.parameters()):,}")

SAE: d_model=1536, d_sae=6144, k=32
Parameters: 18,882,048


In [ ]:
# [5/7] Configurable Dataset Loading
from datasets import load_dataset, Dataset as HFDataset, concatenate_datasets

# Ensure REPO_AVAILABLE is defined even if cell [2b/7] hasn't been run this session
if 'REPO_AVAILABLE' not in dir():
    import os, subprocess
    from pathlib import Path
    _WORK = Path("/content/Deleeuw-AI-x-Bio-hackathon")
    REPO_AVAILABLE = _WORK.exists()

# ======================================================================
# DATASET CONFIGURATION
# ======================================================================
# Option A (DEFAULT): cais/wmdp-corpora — WMDP Machine Unlearning Corpora
#   (Li et al. 2024, arXiv:2403.03218). Publicly available on HuggingFace.
#   Available configs: bio-retain-corpus, cyber-forget-corpus, cyber-retain-corpus
#   NOTE: bio-forget-corpus is NOT publicly available on HuggingFace.
#   This option loads bio-retain-corpus (benign) + local hazard data as fallback.
#
# Option B: SolshineCode/biorefusalaudit-gated — curated eval set
#   Requires HF approval (gated). Use once you have full access (~10K).
#
# Option C: local — reads data/eval_set_public/eval_set_public_v1.jsonl
#   Only available if REPO_AVAILABLE=True (cell [2b/7] must be run first).
#
# Set DATASET_SOURCE = "wmdp" | "gated" | "local"
# ======================================================================

DATASET_SOURCE = "wmdp"  # <-- change to "gated" once biorefusalaudit-gated is accessible

# Guard: local option requires the cloned repo
if DATASET_SOURCE == "local" and not REPO_AVAILABLE:
    print("NOTE: DATASET_SOURCE='local' requested but repo clone failed — switching to 'wmdp'.")
    DATASET_SOURCE = "wmdp"

HF_TEXT_COLUMN  = "prompt"
HF_LABEL_COLUMN = "tier"
dataset = None

if DATASET_SOURCE == "wmdp":
    try:
        print("Loading cais/wmdp-corpora (bio-retain-corpus as benign + local as hazard)...")
        # bio-retain-corpus is publicly available (benign biology data)
        ds_benign = load_dataset("cais/wmdp-corpora", "bio-retain-corpus", split="train")
        ds_benign = ds_benign.map(
            lambda x: {"prompt": x["text"], "tier": "benign_bio"},
            remove_columns=ds_benign.column_names)

        # Hazard data: cais/wmdp-bio-forget-corpus (gated, 24,453 papers, access granted 2026-04-26)
        # Fallback: 22-prompt local eval set if bio-forget-corpus is inaccessible.
        ds_hazard = None
        try:
            print("Loading cais/wmdp-bio-forget-corpus (hazard-adjacent, gated)...")
            _ds_forget_raw = load_dataset("cais/wmdp-bio-forget-corpus", split="train")
            def _fmt_forget(ex):
                # Combine abstract + first 1000 chars of body text for each paper
                chunk = (ex.get("abstract") or "") + " " + (ex.get("text") or "")[:1000]
                return {"prompt": chunk.strip(), "tier": "hazard_adjacent_category"}
            ds_hazard = _ds_forget_raw.map(_fmt_forget, remove_columns=_ds_forget_raw.column_names)
            MAX_HAZARD = 5000
            if len(ds_hazard) > MAX_HAZARD:
                ds_hazard = ds_hazard.shuffle(seed=42).select(range(MAX_HAZARD))
            print(f"Loaded {len(ds_hazard)} hazard-adjacent examples from bio-forget-corpus")
        except Exception as _e:
            print(f"bio-forget-corpus unavailable ({_e}); falling back to local eval set (22 prompts).")
            if REPO_AVAILABLE:
                import json as _json
                _rows = [_json.loads(l) for l in open("data/eval_set_public/eval_set_public_v1.jsonl")]
                _hazard_rows = [r for r in _rows if "hazard" in r.get("tier", "")]
                if _hazard_rows:
                    ds_hazard = HFDataset.from_list([
                        {"prompt": r.get("prompt", ""), "tier": "hazard_adjacent_category"}
                        for r in _hazard_rows
                    ])

        # Balance benign to match hazard pool size
        MAX_PER_TIER = len(ds_hazard) if ds_hazard is not None else 5000
        ds_benign = ds_benign.shuffle(seed=42).select(range(min(MAX_PER_TIER, len(ds_benign))))

        if ds_hazard is not None and len(ds_hazard) > 0:
            dataset = concatenate_datasets([ds_hazard, ds_benign]).shuffle(seed=42)
            print(f"Loaded {len(dataset)} examples: {len(ds_hazard)} hazard-adjacent, {len(ds_benign)} benign-bio")
        else:
            dataset = ds_benign.shuffle(seed=42)
            print(f"Loaded {len(dataset)} benign-bio examples (no hazard data — contrastive loss will be 0)")
        HF_TEXT_COLUMN, HF_LABEL_COLUMN = "prompt", "tier"

    except Exception as e:
        print(f"WMDP-corpora load failed ({e}); falling back to local eval set.")
        DATASET_SOURCE = "local"

elif DATASET_SOURCE == "gated":
    HF_DATASET_REPO = "SolshineCode/biorefusalaudit-gated"
    try:
        print(f"Loading {HF_DATASET_REPO}...")
        dataset = load_dataset(HF_DATASET_REPO, split="train")
        print(f"Loaded {len(dataset)} examples.")
    except Exception as e:
        print(f"Gated dataset unavailable ({e}); falling back to local eval set.")
        DATASET_SOURCE = "local"

if DATASET_SOURCE == "local" or dataset is None:
    if not REPO_AVAILABLE:
        raise RuntimeError(
            "No dataset available: WMDP load failed, gated unavailable, and "
            "repo not cloned (REPO_AVAILABLE=False). "
            "Set GITHUB_TOKEN or HF_TOKEN in Colab Secrets (🔑) and re-run cell [2b/7]."
        )
    import json as _json
    _rows = [_json.loads(l) for l in open("data/eval_set_public/eval_set_public_v1.jsonl")]
    dataset = HFDataset.from_list(_rows)
    HF_TEXT_COLUMN, HF_LABEL_COLUMN = "prompt", "tier"
    print(f"Using local public eval set: {len(dataset)} examples.")
    print("NOTE: 75-prompt fallback active — L_contrastive will plateau; use wmdp or gated option for real training.")

def sample_batch(ds, size=4):
    indices = torch.randint(0, len(ds), (size,))
    return [ds[int(i)] for i in indices]

In [11]:
# [6/7] Training Loop Logic
import torch.nn.functional as F

def compute_contrastive_loss(z_hazard, z_benign, margin=1.0):
    """Encourage separation between hazard-adjacent and benign biological feature profiles."""
    if z_hazard.size(0) == 0 or z_benign.size(0) == 0:
        return torch.tensor(0.0).to(z_hazard.device)

    # Aggregate across sequence tokens
    m_hazard = z_hazard.mean(0)
    m_benign = z_benign.mean(0)

    # Minimize cosine similarity between the two mean feature profiles
    cos_sim = F.cosine_similarity(m_hazard.unsqueeze(0), m_benign.unsqueeze(0))
    return cos_sim.mean()

def track_semantic_drift(sae, tokenizer, features_to_watch=[100, 500, 1000]):
    """Placeholder for semantic drift tracking: identifies top tokens for specific features."""
    # In a real run, this would collect activations across a calibration set.
    return {f"feat_{idx}_drift": 0.0 for idx in features_to_watch}

In [ ]:
# [7/7] Main Execution Loop
import torch.nn.functional as F
from tqdm.auto import tqdm

RUN_TAG = "gemma4-e2b-bio-sae-v2"
HF_OUTPUT_REPO = f"Solshine/{RUN_TAG}"
LR = 3e-4
LAMBDA_SPARSITY = 0.04
LAMBDA_CONTRASTIVE = 0.1
BATCH_SIZE = 4
MAX_STEPS = 5000
SAVE_EVERY = 500

optimizer = torch.optim.AdamW(sae.parameters(), lr=LR)

wandb.init(project="biorefusalaudit-sae-training", name=RUN_TAG,  # v2: bio-forget-corpus, 5000 steps config={
    "model": HF_MODEL, "layer": LAYER, "d_sae": D_SAE, "k": K,
    "lr": LR, "lambda_sparsity": LAMBDA_SPARSITY, "lambda_contrastive": LAMBDA_CONTRASTIVE
})

pbar = tqdm(range(MAX_STEPS))
for step in pbar:
    batch = sample_batch(dataset, BATCH_SIZE)

    all_activations = []
    hazard_indices = []
    benign_indices = []
    offset = 0

    for item in batch:
        prompt = item[HF_TEXT_COLUMN]
        tier = item.get(HF_LABEL_COLUMN, "benign_bio")

        # Wrap with Gemma chat template so the RLHF safety circuit engages (writeup §4.5
        # pass-5 finding: without canonical <start_of_turn>/<end_of_turn> tokens, Gemma 4
        # IT receives input outside its training distribution, refusal-circuit features
        # don't fire, and the resulting SAE features won't transfer to deployment regime).
        # WMDP-corpora documents are wrapped as user messages too — uniform input regime
        # beats mixing chat-formatted eval prompts with raw corpus text.
        if getattr(tokenizer, "chat_template", None):
            msgs = [{"role": "user", "content": prompt}]
            formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        else:
            formatted = prompt

        inputs = tokenizer(
            formatted, return_tensors="pt", truncation=True, max_length=512
        ).to("cuda")

        with torch.no_grad():
            # Run a forward pass (no generation needed) to capture hook activations
            _ = model(**inputs)
            acts = hook.last  # (1, seq_len, d_model)

        seq_len = acts.size(1)
        all_activations.append(acts.squeeze(0))  # (seq_len, d_model)
        token_range = range(offset, offset + seq_len)
        if "hazard" in tier:
            hazard_indices.extend(token_range)
        elif "benign" in tier:
            benign_indices.extend(token_range)
        offset += seq_len

    # Move activations to GPU/Float32 for SAE
    x = torch.cat(all_activations, dim=0).cuda().float()
    x_hat, z, pre_relu = sae(x)

    # Loss 1: Reconstruction (MSE)
    l_recon = F.mse_loss(x_hat, x)

    # Loss 2: Sparsity (L1 on pre-activations)
    l_sparsity = pre_relu.abs().mean()

    # Loss 3: Contrastive (Tier separation)
    l_contrast = torch.tensor(0.0).cuda()
    if hazard_indices and benign_indices:
        l_contrast = compute_contrastive_loss(z[hazard_indices], z[benign_indices])

    total_loss = l_recon + LAMBDA_SPARSITY * l_sparsity + LAMBDA_CONTRASTIVE * l_contrast

    optimizer.zero_grad()
    total_loss.backward()
    sae.project_grad()
    optimizer.step()
    sae.normalize_decoder()

    # Metrics
    l0 = (z > 0).float().sum(-1).mean().item()
    drift = track_semantic_drift(sae, tokenizer)
    log_dict = {
        "step": step,
        "total_loss": total_loss.item(),
        "l_recon": l_recon.item(),
        "l_sparsity": l_sparsity.item(),
        "l_contrastive": l_contrast.item(),
        "l0": l0
    }
    log_dict.update(drift)
    wandb.log(log_dict)
    pbar.set_postfix({"loss": f"{total_loss.item():.4f}", "L0": f"{l0:.1f}"})

    if (step + 1) % SAVE_EVERY == 0:
        torch.save(sae.state_dict(), "sae_weights.pt")
        try:
            api = HfApi()
            api.create_repo(HF_OUTPUT_REPO, exist_ok=True, private=True)
            api.upload_file(
                path_or_fileobj="sae_weights.pt",
                path_in_repo=f"sae_weights_step_{step+1}.pt",
                repo_id=HF_OUTPUT_REPO
            )
            print(f"\nCheckpoint saved to HF: {HF_OUTPUT_REPO}")
        except Exception as e:
            print(f"\nHF Push failed: {e}")

handle.remove()
wandb.finish()

# Final checkpoint — upload regardless of step alignment with SAVE_EVERY
torch.save(sae.state_dict(), "sae_weights_final.pt")
if os.environ.get("HF_TOKEN"):
    try:
        api = HfApi()
        api.create_repo(HF_OUTPUT_REPO, exist_ok=True, private=True)
        api.upload_file(
            path_or_fileobj="sae_weights_final.pt",
            path_in_repo="sae_weights_final.pt",
            repo_id=HF_OUTPUT_REPO
        )
        print(f"Final checkpoint saved to: https://huggingface.co/{HF_OUTPUT_REPO}")
    except Exception as e:
        print(f"Final HF upload failed: {e}")
else:
    print("No HF_TOKEN -- final weights saved locally as sae_weights_final.pt")

print("Training Complete.")